## Setup

In [1]:
from pathlib import Path

# Prefer the new package when available (LangChain deprecation path)
try:
    from langchain_chroma import Chroma
except Exception:
    from langchain_community.vectorstores import Chroma

from langchain_huggingface import HuggingFaceEmbeddings
import os

# Define paths
VECTORSTORE_DIR = Path("../data/vectorstore")

print(f"📂 Vector store location: {VECTORSTORE_DIR.absolute()}")
print(f"📊 Vector store exists: {VECTORSTORE_DIR.exists()}")

/Users/adel/Desktop/Rag-project/rag-env/lib/python3.14/site-packages/langchain_core/_api/deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1
/Users/adel/Desktop/Rag-project/rag-env/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


📂 Vector store location: /Users/adel/Desktop/Rag-project/notebooks/../data/vectorstore
📊 Vector store exists: True


## Retrieval Functions

In [2]:
def load_vectorstore(persist_dir, embedding_model):
    """Load existing vector store from disk.

    embedding_model MUST match ingestion.
    """
    embeddings = HuggingFaceEmbeddings(model_name=embedding_model)

    vectorstore = Chroma(
        embedding_function=embeddings,
        persist_directory=str(persist_dir),
        collection_name="rag",
    )

    return vectorstore


def retrieve_relevant_docs(vectorstore, query, k=3):
    """Retrieve top-k relevant documents for a query"""
    results = vectorstore.similarity_search(query, k=k)
    return results


def format_retrieved_docs(docs):
    """Format retrieved documents for display"""
    formatted = ""
    for i, doc in enumerate(docs, 1):
        formatted += f"\n{'='*60}\n"
        formatted += f"📄 Document {i}:\n"
        formatted += f"Source: {doc.metadata.get('source', 'Unknown')}\n"
        formatted += f"{'-'*60}\n"
        formatted += f"{doc.page_content}\n"
    return formatted

## Load Vector Store

In [ ]:
print("🔄 Loading vector store...")
vectorstore = load_vectorstore(
    persist_dir=VECTORSTORE_DIR,
    # IMPORTANT: must match the model used during ingestion
    embedding_model="sentence-transformers/all-MiniLM-L6-v2",
)

print(f"✅ Vector store loaded!")
print(f"📊 Collection count: {vectorstore._collection.count()}")

🔄 Loading vector store...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 16680.05it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/var/folders/87/zmqs14g943d92b6n0nm7_2qw0000gn/T/ipykernel_27099/2229268834.py:7: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vectorstore = Chroma(


✅ Vector store loaded!
📊 Collection count: 3262


## Test Retrieval

In [4]:
# Test query
test_queries = ["summarize the document?"]

for query in test_queries:
    print(f"\n🔍 Query: '{query}'")
    results = retrieve_relevant_docs(vectorstore, query, k=2)
    print(f"Found {len(results)} relevant chunks:")
    formatted = format_retrieved_docs(results)
    print(formatted)


🔍 Query: 'summarize the document?'
Found 2 relevant chunks:

📄 Document 1:
Source: ../data/documents/200033181_02_dragen-bio-it-platform-v4-2-doc.pdf
------------------------------------------------------------
MAPPING/ALIGNING
SUMMARY
Totalinput
reads
816360354
MAPPING/ALIGNING
SUMMARY
Numberof
duplicate
reads
(marked
not
removed)
15779031 1.93
Document#200033181v02
ForResearchUseOnly.Notforuseindiagnosticprocedures.
326
IlluminaDRAGENBio-ITPlatformv4.2

📄 Document 2:
Source: ../data/documents/200033181_02_dragen-bio-it-platform-v4-2-doc.pdf
------------------------------------------------------------
MAPPING/ALIGNING
SUMMARY
Totalinput
reads
816360354
MAPPING/ALIGNING
SUMMARY
Numberof
duplicate
reads
(marked
not
removed)
15779031 1.93
Document#200033181v02
ForResearchUseOnly.Notforuseindiagnosticprocedures.
326
IlluminaDRAGENBio-ITPlatformv4.2



## Retriever Ready
✅ **Retrieval System Active**

The vector store and retriever are now ready to be used in the RAG pipeline.
- **Next Step**: Run `03_rag_pipeline.ipynb` to integrate with a language model
- **Vectorstore**: Loaded and ready for queries
- **Retrieval Function**: `retrieve_relevant_docs(vectorstore, query, k=3)`